# Research B: train the 12-lead LUDB diffusion model on Kaggle

This is the main training notebook for research B:

- **Clean ECG:** LUDB, 12 leads, 500 Hz, windows of 512 samples.
- **Artifacts:** MIT-BIH NST `bw`, `em`, and `ma`, with three corruptions per clean window.
- **Model:** advanced conditional 1D U-Net with HNF blocks, Bridge/FiLM timestep injection, skip connections, and bottleneck self-attention.
- **Objective:** stable multi-domain loss combining Gaussian-noise L1 and STFT magnitude reconstruction loss.

Validation is split by LUDB record rather than by overlapping windows. Checkpoints include model, EMA, optimizer, scheduler, epoch, configuration, and history so training can continue in another Kaggle session.


## 1. Clone the public repository


In [ ]:
from pathlib import Path
import copy
import gc
import json
import math
import os
import random
import shutil
import subprocess
import sys
import time
import zipfile

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

KAGGLE_WORKING = Path('/kaggle/working')
KAGGLE_TEMP = Path('/kaggle/temp')
KAGGLE_TEMP.mkdir(parents=True, exist_ok=True)
REPO_DIR = KAGGLE_TEMP / 'phase1'
REPO_URL = 'https://github.com/vzyhug/phase1.git'
REPO_BRANCH = 'main'

if not KAGGLE_WORKING.exists():
    raise RuntimeError('This notebook is intended for Kaggle.')

if (REPO_DIR / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_BRANCH, '--depth', '1'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'reset', '--hard', f'origin/{REPO_BRANCH}'], check=True)
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type != 'cuda':
    raise RuntimeError('Enable a GPU accelerator in Kaggle before training.')
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=True)


## 2. Training configuration


In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Research-B data protocol.
TEST_RECORDS = [str(record) for record in range(180, 201)]
VALIDATION_RECORD_FRACTION = 0.20
CHANNELS = 12
WINDOW_LENGTH = 512
TARGET_FS = 500

# Full data by default. Set an integer only for a short diagnostic run.
MAX_TRAIN_SAMPLES = None
MAX_VALIDATION_SAMPLES = 3000

# Research-B model and diffusion.
BASE_FEATS = 80
EMB_DIM = 128
NUM_DIFFUSION_STEPS = 50
BETA_START = 1e-4
BETA_END = 0.5
BETA_SCHEDULE = 'quad'

# Stable multi-domain objective.
LAMBDA_TIME = 1.0
LAMBDA_FREQ = 0.1
STFT_N_FFT = 128
STFT_HOP_LENGTH = 64

# Training. Resume support makes a long run safe across Kaggle sessions.
EPOCHS = 300
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
GRAD_CLIP_NORM = 1.0
EMA_DECAY = 0.999
LR_STEP_SIZE = 150
LR_GAMMA = 0.1
METRIC_VALID_EVERY = 10
MAX_METRIC_VALID_SAMPLES = 256
DDIM_VALID_STEPS = 15
NUM_WORKERS = 0
PIN_MEMORY = False

# For a continuation run, attach the previous Kaggle output as Input.
# Set an explicit path when more than one matching checkpoint is attached.
RESUME_CHECKPOINT = None
AUTO_RESUME_FROM_KAGGLE_INPUT = True

OUTPUT_DIR = KAGGLE_WORKING / 'ludb_research_b'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = {
    'experiment': 'research_b_ludb_12lead_multidomain',
    'data': 'LUDB_12lead_500Hz_512_plus_NST_bw_em_ma',
    'model': 'UNet1D_HNF_Bridge_SelfAttention',
    'loss': 'L1_mean_plus_SNR_weighted_relative_STFT_x0',
    'seed': SEED,
    'test_records': TEST_RECORDS,
    'validation_record_fraction': VALIDATION_RECORD_FRACTION,
    'max_train_samples': MAX_TRAIN_SAMPLES,
    'max_validation_samples': MAX_VALIDATION_SAMPLES,
    'base_feats': BASE_FEATS,
    'emb_dim': EMB_DIM,
    'diffusion_steps': NUM_DIFFUSION_STEPS,
    'beta_start': BETA_START,
    'beta_end': BETA_END,
    'beta_schedule': BETA_SCHEDULE,
    'lambda_time': LAMBDA_TIME,
    'lambda_freq': LAMBDA_FREQ,
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'ema_decay': EMA_DECAY,
}
(OUTPUT_DIR / 'config.json').write_text(json.dumps(CONFIG, indent=2), encoding='utf-8')
print(json.dumps(CONFIG, indent=2))


## 3. Download LUDB and MIT-BIH NST


In [ ]:
from urllib.request import Request, urlopen
from zipfile import ZipFile

DOWNLOAD_DIR = KAGGLE_TEMP / 'downloads'
LUDB_DIR = REPO_DIR / 'data/raw/ludb_database'
NST_DIR = REPO_DIR / 'data/raw/mit_bih_nst'
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

datasets = {
    'ludb': (
        'https://physionet.org/static/published-projects/ludb/'
        'lobachevsky-university-electrocardiography-database-1.0.1.zip',
        DOWNLOAD_DIR / 'ludb.zip', LUDB_DIR,
    ),
    'nst': (
        'https://physionet.org/static/published-projects/nstdb/'
        'mit-bih-noise-stress-test-database-1.0.0.zip',
        DOWNLOAD_DIR / 'nst.zip', NST_DIR,
    ),
}


def download(url, destination):
    if destination.exists() and destination.stat().st_size > 0:
        print('Using cached:', destination.name)
        return
    partial = destination.with_suffix(destination.suffix + '.part')
    partial.unlink(missing_ok=True)
    request = Request(url, headers={'User-Agent': 'Kaggle research-B training'})
    try:
        with urlopen(request, timeout=60) as response, partial.open('wb') as output:
            total = int(response.headers.get('Content-Length', 0)) or None
            with tqdm(total=total, unit='B', unit_scale=True, desc=destination.name) as progress:
                while True:
                    chunk = response.read(1024 * 1024)
                    if not chunk:
                        break
                    output.write(chunk)
                    progress.update(len(chunk))
        partial.replace(destination)
    except Exception:
        partial.unlink(missing_ok=True)
        raise


def extract_flat(archive, destination):
    destination.mkdir(parents=True, exist_ok=True)
    with ZipFile(archive) as zip_file:
        for member in zip_file.infolist():
            if member.is_dir():
                continue
            target = destination / Path(member.filename).name
            if target.exists() and target.stat().st_size == member.file_size:
                continue
            with zip_file.open(member) as source, target.open('wb') as output:
                shutil.copyfileobj(source, output)


for _, (url, archive, destination) in datasets.items():
    download(url, archive)
    extract_flat(archive, destination)

ludb_headers = sorted(LUDB_DIR.glob('*.hea'))
missing_nst = [
    f'{record}.{extension}'
    for record in ('bw', 'em', 'ma')
    for extension in ('hea', 'dat')
    if not (NST_DIR / f'{record}.{extension}').is_file()
]
if len(ludb_headers) != 200:
    raise RuntimeError(f'Expected 200 LUDB records, found {len(ludb_headers)}.')
if missing_nst:
    raise RuntimeError(f'Missing NST files: {missing_nst}')
print('Dataset files verified.')


## 4. Preprocess and synthesize research-B data


In [ ]:
# Run in a child process so the large temporary arrays are released when
# preprocessing completes. The test records are excluded later by label.
test_records_literal = repr(TEST_RECORDS)
preprocess_code = (
    'from src.data_prep.data_preparation import Data_Preparation; '
    f'Data_Preparation(n_type=1, force_rebuild=True, test_records={test_records_literal})'
)
subprocess.run([sys.executable, '-c', preprocess_code], cwd=REPO_DIR, check=True)

CLEAN_PATH = REPO_DIR / 'data/synthesis/clean.npy'
NOISY_PATH = REPO_DIR / 'data/synthesis/noisy.npy'
LABEL_PATH = REPO_DIR / 'data/processed/segments_512/labels.npy'
for required_path in (CLEAN_PATH, NOISY_PATH, LABEL_PATH):
    if not required_path.is_file():
        raise FileNotFoundError(required_path)

clean_memmap = np.load(CLEAN_PATH, mmap_mode='r')
noisy_memmap = np.load(NOISY_PATH, mmap_mode='r')
segment_labels = np.load(LABEL_PATH).astype(str)
if len(clean_memmap) % len(segment_labels) != 0:
    raise RuntimeError('Synthesized samples do not align with LUDB segment labels.')
augmentation_factor = len(clean_memmap) // len(segment_labels)
sample_labels = np.repeat(segment_labels, augmentation_factor)

if clean_memmap.shape != noisy_memmap.shape:
    raise RuntimeError(f'Clean/noisy shape mismatch: {clean_memmap.shape} vs {noisy_memmap.shape}')
if clean_memmap.shape[1:] != (WINDOW_LENGTH, CHANNELS):
    raise RuntimeError(f'Expected (N, 512, 12), received {clean_memmap.shape}')
if len(sample_labels) != len(clean_memmap):
    raise RuntimeError('Repeated labels do not match synthesized sample count.')

print('Clean/noisy:', clean_memmap.shape)
print('Augmentation factor:', augmentation_factor)
print('Approximate clean+noisy disk size:',
      round((CLEAN_PATH.stat().st_size + NOISY_PATH.stat().st_size) / 1024**3, 2), 'GB')


## 5. Split train, validation, and test by record

Windows from one LUDB record overlap heavily. Splitting individual windows would leak nearly identical ECG into train and validation, so this notebook first assigns whole records to a split and only then selects their windows.


In [ ]:
all_record_ids = sorted(np.unique(sample_labels), key=lambda value: int(value))
development_records = [record for record in all_record_ids if record not in set(TEST_RECORDS)]
train_records, validation_records = train_test_split(
    development_records,
    test_size=VALIDATION_RECORD_FRACTION,
    random_state=SEED,
    shuffle=True,
)

train_indices = np.flatnonzero(np.isin(sample_labels, train_records))
validation_indices = np.flatnonzero(np.isin(sample_labels, validation_records))
test_indices = np.flatnonzero(np.isin(sample_labels, TEST_RECORDS))

rng = np.random.default_rng(SEED)
if MAX_TRAIN_SAMPLES is not None and len(train_indices) > MAX_TRAIN_SAMPLES:
    train_indices = np.sort(rng.choice(train_indices, MAX_TRAIN_SAMPLES, replace=False))
if MAX_VALIDATION_SAMPLES is not None and len(validation_indices) > MAX_VALIDATION_SAMPLES:
    validation_indices = np.sort(rng.choice(validation_indices, MAX_VALIDATION_SAMPLES, replace=False))

split_sets = [set(train_records), set(validation_records), set(TEST_RECORDS)]
if split_sets[0] & split_sets[1] or split_sets[0] & split_sets[2] or split_sets[1] & split_sets[2]:
    raise RuntimeError('Record leakage detected between train/validation/test.')
if min(len(train_indices), len(validation_indices), len(test_indices)) == 0:
    raise RuntimeError('At least one data split is empty.')

split_manifest = {
    'train_records': sorted(train_records, key=int),
    'validation_records': sorted(validation_records, key=int),
    'test_records': sorted(TEST_RECORDS, key=int),
    'train_samples': int(len(train_indices)),
    'validation_samples': int(len(validation_indices)),
    'test_samples': int(len(test_indices)),
}
(OUTPUT_DIR / 'split_manifest.json').write_text(json.dumps(split_manifest, indent=2), encoding='utf-8')
print(json.dumps(split_manifest, indent=2))


## 6. Memory-efficient data loaders


In [ ]:
class MemoryMappedECGDataset(Dataset):
    def __init__(self, clean_path, noisy_path, indices):
        self.clean = np.load(clean_path, mmap_mode='r')
        self.noisy = np.load(noisy_path, mmap_mode='r')
        self.indices = np.asarray(indices, dtype=np.int64)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        index = int(self.indices[item])
        # Copy one sample to obtain writable contiguous tensors without
        # materializing the complete multi-GB dataset in RAM.
        clean = torch.from_numpy(np.array(self.clean[index], dtype=np.float32, copy=True)).permute(1, 0)
        noisy = torch.from_numpy(np.array(self.noisy[index], dtype=np.float32, copy=True)).permute(1, 0)
        return clean, noisy


train_dataset = MemoryMappedECGDataset(CLEAN_PATH, NOISY_PATH, train_indices)
validation_dataset = MemoryMappedECGDataset(CLEAN_PATH, NOISY_PATH, validation_indices)
loader_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, generator=loader_generator,
)
validation_loader = DataLoader(
    validation_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
)

sample_clean, sample_noisy = train_dataset[0]
print('Sample tensors:', tuple(sample_clean.shape), tuple(sample_noisy.shape))
print('Finite:', bool(torch.isfinite(sample_clean).all() and torch.isfinite(sample_noisy).all()))
print('Train batches/epoch:', len(train_loader), '| Validation batches:', len(validation_loader))


## 7. Research-B U-Net and stable multi-domain DDPM


In [ ]:
from src.models.unet_1d import UNet1D
from src.models.main_model import DDPM


def relative_stft_loss(prediction, target, continuous_sqrt_alpha):
    batch, channels, length = prediction.shape
    prediction_flat = prediction.reshape(batch * channels, length)
    target_flat = target.reshape(batch * channels, length)
    window = torch.hann_window(STFT_N_FFT, device=prediction.device)
    prediction_stft = torch.stft(
        prediction_flat.float(), n_fft=STFT_N_FFT,
        hop_length=STFT_HOP_LENGTH, window=window, return_complex=True,
    )
    target_stft = torch.stft(
        target_flat.float(), n_fft=STFT_N_FFT,
        hop_length=STFT_HOP_LENGTH, window=window, return_complex=True,
    )
    prediction_mag = prediction_stft.abs().reshape(batch, channels, *prediction_stft.shape[-2:])
    target_mag = target_stft.abs().reshape(batch, channels, *target_stft.shape[-2:])
    reduce_dims = (1, 2, 3)
    spectral_error = ((prediction_mag - target_mag) ** 2).mean(dim=reduce_dims)
    target_power = (target_mag ** 2).mean(dim=reduce_dims).clamp_min(1e-6)
    relative_error = spectral_error / target_power

    alpha_bar = continuous_sqrt_alpha.reshape(batch) ** 2
    diffusion_snr = alpha_bar / torch.clamp(1.0 - alpha_bar, min=1e-6)
    frequency_weight = torch.clamp(diffusion_snr, max=1.0)
    return (relative_error * frequency_weight).mean()


class StableResearchBDDPM(DDPM):
    def training_losses(self, clean, noisy_condition):
        batch = clean.shape[0]
        timestep = torch.randint(0, self.num_steps, (batch,), device=clean.device)
        boundaries = torch.as_tensor(
            self.sqrt_alphas_cumprod_prev, dtype=clean.dtype, device=clean.device
        )
        lower = boundaries[timestep]
        upper = boundaries[timestep + 1]
        continuous = lower + torch.rand(batch, device=clean.device) * (upper - lower)
        continuous_3d = continuous.view(batch, 1, 1)

        gaussian_noise = torch.randn_like(clean)
        residual_scale = torch.sqrt(torch.clamp(1.0 - continuous_3d ** 2, min=0.0))
        x_t = continuous_3d * clean + residual_scale * gaussian_noise
        predicted_noise = self.model(x_t, noisy_condition, continuous.view(batch, 1))

        time_loss = F.l1_loss(predicted_noise, gaussian_noise, reduction='mean')
        x0_prediction = (x_t - residual_scale * predicted_noise) / continuous_3d.clamp_min(1e-6)
        frequency_loss = relative_stft_loss(x0_prediction, clean, continuous)
        total_loss = LAMBDA_TIME * time_loss + LAMBDA_FREQ * frequency_loss
        return {
            'total': total_loss,
            'time': time_loss,
            'frequency': frequency_loss,
        }

    def forward(self, clean, noisy_condition):
        return self.training_losses(clean, noisy_condition)['total']


base_model = UNet1D(
    in_channels=CHANNELS * 2,
    base_channels=BASE_FEATS,
    emb_dim=EMB_DIM,
    out_channels=CHANNELS,
).to(DEVICE)

ddpm_config = {
    'train': {'lambda_time': LAMBDA_TIME, 'lambda_freq': LAMBDA_FREQ},
    'diffusion': {
        'num_steps': NUM_DIFFUSION_STEPS,
        'beta_start': BETA_START,
        'beta_end': BETA_END,
        'schedule': BETA_SCHEDULE,
    },
}
model = StableResearchBDDPM(base_model, ddpm_config, DEVICE, conditional=True).to(DEVICE)
ema_model = copy.deepcopy(model).eval()
for parameter in ema_model.parameters():
    parameter.requires_grad_(False)

parameter_count = sum(parameter.numel() for parameter in model.parameters())
print(f'Model parameters: {parameter_count:,}')


## 8. Metrics, EMA, and validation


In [ ]:
@torch.no_grad()
def update_ema(ema, current, decay):
    ema_parameters = dict(ema.named_parameters())
    for name, parameter in current.named_parameters():
        ema_parameters[name].mul_(decay).add_(parameter, alpha=1.0 - decay)
    ema_buffers = dict(ema.named_buffers())
    for name, buffer in current.named_buffers():
        if name in ema_buffers:
            ema_buffers[name].copy_(buffer)


def waveform_metrics(clean, noisy, denoised):
    clean_flat = clean.reshape(clean.shape[0], -1)
    noisy_flat = noisy.reshape(noisy.shape[0], -1)
    denoised_flat = denoised.reshape(denoised.shape[0], -1)
    signal_power = torch.sum(clean_flat ** 2, dim=1)
    input_error = torch.sum((noisy_flat - clean_flat) ** 2, dim=1)
    output_error = torch.sum((denoised_flat - clean_flat) ** 2, dim=1)
    cosine = F.cosine_similarity(clean_flat, denoised_flat, dim=1)
    snr_in = 10.0 * torch.log10((signal_power + 1e-8) / (input_error + 1e-8))
    snr_out = 10.0 * torch.log10((signal_power + 1e-8) / (output_error + 1e-8))
    return cosine, snr_in, snr_out


@torch.no_grad()
def validate(validation_model, compute_sampling_metrics):
    validation_model.eval()
    totals, times, frequencies = [], [], []
    cosines, snr_inputs, snr_outputs = [], [], []
    sampled = 0

    for clean, noisy in tqdm(validation_loader, desc='validation', leave=False):
        clean = clean.to(DEVICE)
        noisy = noisy.to(DEVICE)
        losses = validation_model.training_losses(clean, noisy)
        totals.append(losses['total'].item())
        times.append(losses['time'].item())
        frequencies.append(losses['frequency'].item())

        if compute_sampling_metrics and sampled < MAX_METRIC_VALID_SAMPLES:
            remaining = MAX_METRIC_VALID_SAMPLES - sampled
            metric_clean = clean[:remaining]
            metric_noisy = noisy[:remaining]
            denoised = validation_model.denoising(
                metric_noisy, use_ddim=True, ddim_steps=DDIM_VALID_STEPS,
                ddim_eta=0.0, num_shots=1,
            )
            cosine, snr_in, snr_out = waveform_metrics(metric_clean, metric_noisy, denoised)
            cosines.extend(cosine.cpu().tolist())
            snr_inputs.extend(snr_in.cpu().tolist())
            snr_outputs.extend(snr_out.cpu().tolist())
            sampled += len(metric_clean)

    return {
        'val_total': float(np.mean(totals)),
        'val_time': float(np.mean(times)),
        'val_frequency': float(np.mean(frequencies)),
        'val_cosine': float(np.mean(cosines)) if cosines else np.nan,
        'val_snr_in': float(np.mean(snr_inputs)) if snr_inputs else np.nan,
        'val_snr_out': float(np.mean(snr_outputs)) if snr_outputs else np.nan,
        'val_snr_improvement': (
            float(np.mean(snr_outputs) - np.mean(snr_inputs)) if snr_outputs else np.nan
        ),
    }


## 9. Resume setup


In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer, step_size=LR_STEP_SIZE, gamma=LR_GAMMA
)
start_epoch = 1
best_validation_loss = float('inf')
history = []


def find_resume_checkpoint():
    if RESUME_CHECKPOINT:
        return Path(RESUME_CHECKPOINT)
    if not AUTO_RESUME_FROM_KAGGLE_INPUT:
        return None
    candidates = sorted(Path('/kaggle/input').glob('**/ludb_research_b_last_resume.pth'))
    if len(candidates) > 1:
        raise RuntimeError(
            'Multiple resume checkpoints found. Set RESUME_CHECKPOINT explicitly:\n'
            + '\n'.join(str(path) for path in candidates)
        )
    return candidates[0] if candidates else None


resume_path = find_resume_checkpoint()
if resume_path is not None:
    if not resume_path.is_file():
        raise FileNotFoundError(resume_path)
    checkpoint = torch.load(resume_path, map_location=DEVICE, weights_only=False)
    checkpoint_config = checkpoint.get('config', {})
    critical_keys = ('base_feats', 'emb_dim', 'diffusion_steps', 'beta_start', 'beta_end', 'beta_schedule')
    mismatches = {
        key: (checkpoint_config.get(key), CONFIG.get(key))
        for key in critical_keys if checkpoint_config.get(key) != CONFIG.get(key)
    }
    if mismatches:
        raise RuntimeError(f'Resume configuration mismatch: {mismatches}')
    model.load_state_dict(checkpoint['model_state_dict'])
    ema_model.load_state_dict(checkpoint['ema_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    random.setstate(checkpoint['python_rng_state'])
    np.random.set_state(checkpoint['numpy_rng_state'])
    torch.set_rng_state(checkpoint['torch_rng_state'])
    torch.cuda.set_rng_state_all(checkpoint['cuda_rng_state_all'])
    loader_generator.set_state(checkpoint['loader_generator_state'])
    start_epoch = int(checkpoint['epoch']) + 1
    best_validation_loss = float(checkpoint['best_validation_loss'])
    history = list(checkpoint.get('history', []))
    print(f'Resuming from epoch {start_epoch}: {resume_path}')
else:
    print('Starting a new training run.')


## 10. Train and checkpoint every epoch


In [ ]:
LAST_RESUME_PATH = OUTPUT_DIR / 'ludb_research_b_last_resume.pth'
BEST_RESUME_PATH = OUTPUT_DIR / 'ludb_research_b_best_resume.pth'
BEST_INFERENCE_PATH = OUTPUT_DIR / 'model.pth'
FINAL_INFERENCE_PATH = OUTPUT_DIR / 'final.pth'
LOG_PATH = OUTPUT_DIR / 'training_log.csv'


def atomic_torch_save(payload, path):
    temporary_path = path.with_suffix(path.suffix + '.tmp')
    torch.save(payload, temporary_path)
    temporary_path.replace(path)


def save_resume_checkpoint(path, epoch):
    payload = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'ema_state_dict': ema_model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_validation_loss': best_validation_loss,
        'history': history,
        'config': CONFIG,
        'split_manifest': split_manifest,
        'python_rng_state': random.getstate(),
        'numpy_rng_state': np.random.get_state(),
        'torch_rng_state': torch.get_rng_state(),
        'cuda_rng_state_all': torch.cuda.get_rng_state_all(),
        'loader_generator_state': loader_generator.get_state(),
    }
    atomic_torch_save(payload, path)


for epoch in range(start_epoch, EPOCHS + 1):
    epoch_start = time.time()
    model.train()
    train_total, train_time, train_frequency = [], [], []

    for clean, noisy in tqdm(train_loader, desc=f'epoch {epoch}/{EPOCHS}', leave=False):
        clean = clean.to(DEVICE, non_blocking=True)
        noisy = noisy.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        losses = model.training_losses(clean, noisy)
        if not torch.isfinite(losses['total']):
            raise FloatingPointError(
                f'Non-finite loss at epoch {epoch}. The current checkpoint must not be evaluated.'
            )
        losses['total'].backward()
        gradient_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        if not torch.isfinite(gradient_norm):
            raise FloatingPointError(f'Non-finite gradient at epoch {epoch}.')
        optimizer.step()
        update_ema(ema_model, model, EMA_DECAY)

        train_total.append(losses['total'].item())
        train_time.append(losses['time'].item())
        train_frequency.append(losses['frequency'].item())

    scheduler.step()
    metric_epoch = epoch == 1 or epoch % METRIC_VALID_EVERY == 0
    validation_devices = [torch.cuda.current_device()]
    with torch.random.fork_rng(devices=validation_devices):
        torch.manual_seed(SEED + 10000)
        torch.cuda.manual_seed_all(SEED + 10000)
        validation_metrics = validate(ema_model, metric_epoch)

    row = {
        'epoch': epoch,
        'lr': optimizer.param_groups[0]['lr'],
        'train_total': float(np.mean(train_total)),
        'train_time': float(np.mean(train_time)),
        'train_frequency': float(np.mean(train_frequency)),
        **validation_metrics,
        'seconds': round(time.time() - epoch_start, 2),
    }
    history.append(row)
    print(row)

    current_validation_loss = row['val_total']
    if not np.isfinite(current_validation_loss):
        raise FloatingPointError('Validation loss is not finite; checkpoint was not saved.')

    improved = current_validation_loss < best_validation_loss
    if improved:
        best_validation_loss = current_validation_loss

    # Save after updating best_validation_loss so resume metadata is exact.
    save_resume_checkpoint(LAST_RESUME_PATH, epoch)
    if improved:
        save_resume_checkpoint(BEST_RESUME_PATH, epoch)
        atomic_torch_save(ema_model.state_dict(), BEST_INFERENCE_PATH)
        print('Saved new best:', BEST_INFERENCE_PATH)

    pd.DataFrame(history).to_csv(LOG_PATH, index=False)
    torch.cuda.empty_cache()
    gc.collect()

atomic_torch_save(ema_model.state_dict(), FINAL_INFERENCE_PATH)
print('Training complete.')


## 11. Curves and checkpoint sanity check


In [ ]:
history_frame = pd.DataFrame(history)
display(history_frame.tail())

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(history_frame['epoch'], history_frame['train_total'], label='train')
axes[0].plot(history_frame['epoch'], history_frame['val_total'], label='validation')
axes[0].set_title('Multi-domain total loss')
axes[1].plot(history_frame['epoch'], history_frame['train_time'], label='time')
axes[1].plot(history_frame['epoch'], history_frame['train_frequency'], label='frequency')
axes[1].set_title('Training loss components')
axes[2].plot(history_frame['epoch'], history_frame['val_cosine'], label='cosine')
axes[2].plot(history_frame['epoch'], history_frame['val_snr_improvement'], label='SNR improvement')
axes[2].set_title('Sampled validation metrics')
for axis in axes:
    axis.set_xlabel('Epoch')
    axis.grid(alpha=0.25)
    axis.legend()
plt.tight_layout()
curve_path = OUTPUT_DIR / 'training_curves.png'
plt.savefig(curve_path, dpi=160, bbox_inches='tight')
plt.show()

if not BEST_INFERENCE_PATH.is_file():
    raise RuntimeError('No best model.pth was produced.')
state_dict = torch.load(BEST_INFERENCE_PATH, map_location='cpu')
expected_input = state_dict['model.enc1.multi_convs.0.weight'].shape[1]
expected_output = state_dict['model.final.weight'].shape[0]
print('Checkpoint architecture: input channels =', expected_input, '| output channels =', expected_output)
if expected_input != 24 or expected_output != 12:
    raise RuntimeError('Checkpoint is not the research-B 12-lead U-Net.')


## 12. Package Kaggle Output


In [ ]:
archive_path = KAGGLE_WORKING / 'ludb_research_b_outputs.zip'
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for output_path in sorted(OUTPUT_DIR.glob('*')):
        archive.write(output_path, arcname=output_path.name)

print('Output directory:', OUTPUT_DIR)
for output_path in sorted(OUTPUT_DIR.glob('*')):
    print(output_path.name, f'{output_path.stat().st_size / 1024**2:.2f} MB')
print('Archive:', archive_path)

from IPython.display import FileLink, display
display(FileLink(str(BEST_INFERENCE_PATH)))
display(FileLink(str(LAST_RESUME_PATH)))
display(FileLink(str(archive_path)))


## 13. Continuing in another Kaggle session

Publish the completed or timed-out Kaggle version, add that version's Output as an Input to the next session, and run this notebook again. It automatically finds a single `ludb_research_b_last_resume.pth` under `/kaggle/input` and continues from the next epoch. The dataset is regenerated deterministically, so record splits and synthesized artifacts remain reproducible.

Use `model.pth` for inference and final evaluation. Use `ludb_research_b_last_resume.pth` only to continue training.
